# Text Summarization Pipeline — CNN/DailyMail Dataset

## Problem Statement & Approach

---

**Task:** Text Summarization on the CNN/DailyMail news dataset.

**Dataset:** The CNN/DailyMail dataset is a large-scale collection of news articles paired with
multi-sentence highlights (human-written summaries). Each sample has:
- `id`: Unique article identifier
- `article`: The full news article text
- `highlights`: Human-written bullet-point summary

**Goal:** Given a news article, produce a concise summary that captures the key information.

**Approach — Three distinct modeling strategies:**

| # | Model | Type | Description |
|---|-------|------|-------------|
| 1 | **RandomForest** | ML (Extractive) | Uses TF-IDF statistics, sentence position, and length features to classify which sentences belong in the summary |
| 2 | **Seq2Seq GRU + Attention** | DL (Abstractive) | Encoder-Decoder neural network with Bahdanau attention, trained from scratch to generate summaries word-by-word |
| 3 | **T5-small** | Pre-trained (Abstractive) | Google's T5-small from Hugging Face with hierarchical chunking for long documents |

**Evaluation:** All models are compared using ROUGE-1, ROUGE-2, and ROUGE-L metrics on the same test set.

### Importing Modules

---

Used Modules:
1. Pandas
2. Numpy
3. re
4. os, time, random
5. nltk
6. Sklearn (TfidfVectorizer, RandomForestClassifier)
7. PyTorch (GRU Seq2Seq with Attention)
8. Hugging Face (T5 pretrained model)
9. rouge-score
10. matplotlib & seaborn
11. Gradio (UI)
12. FastAPI & uvicorn (API)

In [ ]:
# Install required packages (uncomment if needed)
# !pip install transformers sentencepiece rouge-score gradio fastapi uvicorn nltk tqdm

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import time
import random
import warnings
warnings.filterwarnings('ignore')

# ML Modules
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, recall_score)

# DL Modules Using PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Hugging Face Modules
from transformers import T5Tokenizer, T5ForConditionalGeneration

# ROUGE evaluation
from rouge_score import rouge_scorer

# Visualizing
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('dark_background')

# Progress bar
from tqdm.auto import tqdm

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print("All modules imported successfully!")

### Configuration

---

All hyperparameters and constants defined in one place.

In [ ]:
# ============================================================
# Global Configuration
# ============================================================

# Dataset sizes
TRAIN_SIZE = 20000   # Number of training samples to load
VAL_SIZE   = 2000    # Number of validation samples
TEST_SIZE  = 4000    # Number of test samples

# ML Model config
ML_N_ESTIMATORS = 200    # RandomForest trees
ML_N_SENTENCES  = 3      # Number of sentences to extract

# DL Model config
DL_TRAIN_SIZE       = 10000   # Subset for DL training (faster)
MAX_ARTICLE_TOKENS  = 100     # Max tokens per article (DL)
MAX_SUMMARY_TOKENS  = 30      # Max tokens per summary (DL)
DL_EMBED_DIM        = 128
DL_HIDDEN_DIM       = 256
DL_DROPOUT          = 0.3
DL_BATCH_SIZE       = 64
DL_EPOCHS           = 3
DL_LR               = 0.001
VOCAB_FREQ_THRESHOLD = 3      # Min word frequency for vocabulary

# Special token indices
PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
UNK_IDX = 3

# Evaluation
N_EVAL_SAMPLES = 200  # Samples to evaluate each model on

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### Loading the CNN/DailyMail Dataset

---

Loading subsets from the CSV files for efficient processing.
- **Train:** 20,000 articles
- **Validation:** 2,000 articles
- **Test:** 4,000 articles

In [ ]:
# ============================================================
# Load datasets from CSV files
# ============================================================

print("Loading datasets...")
train_df = pd.read_csv('cnn_dailymail/train.csv', nrows=TRAIN_SIZE)
val_df   = pd.read_csv('cnn_dailymail/validation.csv', nrows=VAL_SIZE)
test_df  = pd.read_csv('cnn_dailymail/test.csv', nrows=TEST_SIZE)

print(f"Training set:   {train_df.shape}")
print(f"Validation set: {val_df.shape}")
print(f"Test set:       {test_df.shape}")

print("\n--- Columns ---")
print(train_df.columns.tolist())

print("\n--- Data Types ---")
print(train_df.dtypes)

print("\n--- Missing Values ---")
print(train_df.isnull().sum())

### Exploratory Data Analysis (EDA)

---

Understanding the dataset: text lengths, distributions, and sample data.

In [ ]:
# ============================================================
# EDA — Dataset Statistics
# ============================================================

print("=" * 60)
print("DATASET EXPLORATION")
print("=" * 60)

# Compute word counts
train_df['article_word_count']    = train_df['article'].apply(lambda x: len(str(x).split()))
train_df['highlights_word_count'] = train_df['highlights'].apply(lambda x: len(str(x).split()))

print("\n--- Article Word Count Statistics ---")
print(train_df['article_word_count'].describe().to_string())

print("\n--- Highlights Word Count Statistics ---")
print(train_df['highlights_word_count'].describe().to_string())

# Compression ratio
compression = train_df['highlights_word_count'] / train_df['article_word_count']
print(f"\n--- Compression Ratio (summary/article) ---")
print(f"  Mean:   {compression.mean():.4f}")
print(f"  Median: {compression.median():.4f}")

print("\n--- Sample Article (first 500 chars) ---")
print(train_df['article'].iloc[0][:500])
print("\n--- Corresponding Highlights ---")
print(train_df['highlights'].iloc[0])

In [ ]:
# ============================================================
# EDA — Visualizations
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Article word count distribution
axes[0, 0].hist(train_df['article_word_count'], bins=50,
                color='#00d4aa', alpha=0.8, edgecolor='white')
axes[0, 0].set_title('Article Word Count Distribution', fontweight='bold', fontsize=12)
axes[0, 0].set_xlabel('Word Count')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(train_df['article_word_count'].mean(), color='#ff6b6b',
                   linestyle='--', linewidth=2,
                   label=f"Mean: {train_df['article_word_count'].mean():.0f}")
axes[0, 0].legend()

# 2. Highlights word count distribution
axes[0, 1].hist(train_df['highlights_word_count'], bins=50,
                color='#ff6b6b', alpha=0.8, edgecolor='white')
axes[0, 1].set_title('Highlights Word Count Distribution', fontweight='bold', fontsize=12)
axes[0, 1].set_xlabel('Word Count')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(train_df['highlights_word_count'].mean(), color='#00d4aa',
                   linestyle='--', linewidth=2,
                   label=f"Mean: {train_df['highlights_word_count'].mean():.0f}")
axes[0, 1].legend()

# 3. Compression ratio
compression = train_df['highlights_word_count'] / train_df['article_word_count']
axes[1, 0].hist(compression, bins=50, color='#4ecdc4', alpha=0.8, edgecolor='white')
axes[1, 0].set_title('Compression Ratio (Summary / Article)', fontweight='bold', fontsize=12)
axes[1, 0].set_xlabel('Ratio')
axes[1, 0].set_ylabel('Frequency')

# 4. Box plot comparison
bp = axes[1, 1].boxplot(
    [train_df['article_word_count'], train_df['highlights_word_count']],
    labels=['Articles', 'Highlights'],
    patch_artist=True,
    medianprops=dict(color='#ff6b6b', linewidth=2)
)
bp['boxes'][0].set_facecolor('#00d4aa')
bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#ff6b6b')
bp['boxes'][1].set_alpha(0.7)
axes[1, 1].set_title('Article vs Highlights Length', fontweight='bold', fontsize=12)
axes[1, 1].set_ylabel('Word Count')

plt.suptitle('CNN/DailyMail Dataset — Exploratory Analysis',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Data Preprocessing Pipeline

---

**Steps performed:**

1. **Handle missing data:** Drop any rows with NaN values in article or highlights
2. **Text cleaning:** Remove special characters, normalize whitespace, lowercase for processing
3. **Sentence tokenization:** Split articles into individual sentences (for ML extractive model)
4. **Word tokenization:** Split text into words (for DL model vocabulary building)
5. **Feature engineering:** Compute TF-IDF, positional, and length features (for ML model)
6. **Sequence encoding:** Convert text to numerical sequences with padding (for DL model)
7. **T5 tokenizer:** The Hugging Face tokenizer handles its own preprocessing automatically

> Each model requires different preprocessing — the pipeline adapts to each approach.

In [ ]:
# ============================================================
# Data Preprocessing — Text Cleaning
# ============================================================

def clean_text(text):
    """Clean and normalize text for model consumption."""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text.strip())
    # Remove non-printable characters but keep punctuation
    text = re.sub(r'[^\w\s.,!?;:\'\"-]', '', text)
    return text

# Apply cleaning to all datasets
print("Cleaning text data...")
for df in [train_df, val_df, test_df]:
    df['article_clean']    = df['article'].apply(clean_text)
    df['highlights_clean'] = df['highlights'].apply(clean_text)

# Drop rows with empty text
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    before = len(df)
    df.dropna(subset=['article_clean', 'highlights_clean'], inplace=True)
    df = df[df['article_clean'].str.len() > 0]
    after = len(df)
    if before != after:
        print(f"  {name}: dropped {before - after} empty rows")

print(f"\nAfter cleaning:")
print(f"  Train: {len(train_df):,} articles")
print(f"  Val:   {len(val_df):,} articles")
print(f"  Test:  {len(test_df):,} articles")

print("\n--- Sample Cleaned Article ---")
print(train_df['article_clean'].iloc[0][:300])

## Model 1 — Machine Learning: Extractive Summarization (RandomForest)

---

**Approach:** Frame summarization as a **sentence classification** problem.

For each sentence in an article:
- Extract features: TF-IDF statistics, position in document, length, etc.
- Label: Does this sentence appear in the reference summary? (binary)
- Train a RandomForest classifier on these labeled sentence features

At inference: score all sentences in a new article, select the top-N highest-scoring sentences, and return them in their original order as the extractive summary.

**Features used:**
| Feature | Description |
|---------|-------------|
| `position` | Normalized position of sentence in the article (0 = first, 1 = last) |
| `length` | Number of words in the sentence |
| `is_first` | Whether this is the first sentence |
| `is_last` | Whether this is the last sentence |
| `tfidf_mean` | Mean TF-IDF score across words in the sentence |
| `tfidf_max` | Maximum TF-IDF score of any word in the sentence |
| `tfidf_sum` | Sum of TF-IDF scores for the sentence |
| `has_number` | Whether the sentence contains a number |

### ML — Feature Engineering & Label Generation

In [ ]:
# ============================================================
# ML Model — Helper Functions
# ============================================================

# Load stopwords
stop_words = set(stopwords.words('english'))

def label_sentences(sentences, highlights_text):
    """
    Label each sentence: 1 if significant word overlap with highlights, else 0.
    This creates the ground truth for the sentence classifier.
    """
    # Extract content words from highlights
    highlight_words = set(
        w.lower() for w in highlights_text.split()
        if w.lower() not in stop_words and len(w) > 2
    )

    labels = []
    for sent in sentences:
        sent_words = set(
            w.lower() for w in sent.split()
            if w.lower() not in stop_words and len(w) > 2
        )
        if len(sent_words) == 0:
            labels.append(0)
            continue
        # Compute word overlap ratio
        overlap = len(sent_words & highlight_words) / len(sent_words)
        labels.append(1 if overlap >= 0.3 else 0)
    return labels


print("Helper functions defined.")

In [ ]:
# ============================================================
# ML Model — Build Training Dataset (sentence-level)
# ============================================================

print("Building ML training dataset...")
print("This involves: sentence splitting → labeling → TF-IDF → feature extraction")
print()

# Step 1: Split all articles into sentences, compute labels
sentence_records = []

for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Processing articles"):
    article = row['article_clean']
    highlights = row['highlights_clean']

    sentences = sent_tokenize(article)
    if len(sentences) < 2:
        continue

    labels = label_sentences(sentences, highlights)
    n = len(sentences)

    for i, (sent, label) in enumerate(zip(sentences, labels)):
        words = sent.split()
        sentence_records.append({
            'sentence': sent,
            'label': label,
            'position': i / max(n - 1, 1),
            'length': len(words),
            'is_first': int(i == 0),
            'is_last': int(i == n - 1),
            'has_number': int(bool(re.search(r'\d', sent))),
        })

sent_df = pd.DataFrame(sentence_records)
print(f"\nTotal sentences extracted: {len(sent_df):,}")
print(f"Positive (in summary):    {sent_df['label'].sum():,} ({sent_df['label'].mean()*100:.1f}%)")
print(f"Negative (not in summary): {(1-sent_df['label']).sum():,.0f}")

In [ ]:
# ============================================================
# ML Model — TF-IDF Feature Computation
# ============================================================

# Fit a global TF-IDF vectorizer on all training sentences
print("Fitting global TF-IDF vectorizer on all sentences...")
tfidf_vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(sent_df['sentence'])

# Add TF-IDF summary statistics as features
print("Computing TF-IDF statistics per sentence...")
sent_df['tfidf_mean'] = np.array(tfidf_matrix.mean(axis=1)).flatten()
sent_df['tfidf_max']  = np.array(tfidf_matrix.max(axis=1).toarray()).flatten()
sent_df['tfidf_sum']  = np.array(tfidf_matrix.sum(axis=1)).flatten()

# Define feature columns
FEATURE_COLS = ['position', 'length', 'is_first', 'is_last',
                'tfidf_mean', 'tfidf_max', 'tfidf_sum', 'has_number']

X_train_ml = sent_df[FEATURE_COLS].values
y_train_ml = sent_df['label'].values

print(f"\nFeature matrix shape: {X_train_ml.shape}")
print(f"Label distribution:   {np.bincount(y_train_ml)}")
print(f"Features: {FEATURE_COLS}")

### ML — Training the RandomForest Classifier

In [ ]:
# ============================================================
# ML Model — Train RandomForest
# ============================================================

print("Training RandomForest classifier...")
ml_train_start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=ML_N_ESTIMATORS,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',  # Handle class imbalance
    random_state=SEED,
    n_jobs=-1,
    verbose=0
)

rf_model.fit(X_train_ml, y_train_ml)

ml_train_time = time.time() - ml_train_start
print(f"ML Training completed in {ml_train_time:.1f} seconds")

# Feature importance
importances = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n--- Feature Importance ---")
print(importances.to_string(index=False))

# Quick training accuracy
train_pred = rf_model.predict(X_train_ml)
print(f"\nTraining Accuracy: {accuracy_score(y_train_ml, train_pred):.4f}")
print(f"Training F1-Score: {f1_score(y_train_ml, train_pred):.4f}")

In [ ]:
# ============================================================
# ML Model — Feature Importance Visualization
# ============================================================

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(importances)))
bars = ax.barh(importances['Feature'], importances['Importance'], color=colors, edgecolor='white')
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('RandomForest — Feature Importance', fontsize=14, fontweight='bold')
ax.invert_yaxis()

# Add value labels
for bar, val in zip(bars, importances['Importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### ML — Summarization Function & Evaluation

In [ ]:
# ============================================================
# ML Model — Summarization Function
# ============================================================

def ml_summarize(article, model=rf_model, vectorizer=tfidf_vectorizer, n_sentences=ML_N_SENTENCES):
    """
    Extractive summarization using the trained RandomForest.
    Scores each sentence and selects the top-N, maintaining original order.
    """
    sentences = sent_tokenize(article)
    if len(sentences) <= n_sentences:
        return ' '.join(sentences)

    n = len(sentences)
    features = []
    for i, sent in enumerate(sentences):
        words = sent.split()
        features.append({
            'position': i / max(n - 1, 1),
            'length': len(words),
            'is_first': int(i == 0),
            'is_last': int(i == n - 1),
            'has_number': int(bool(re.search(r'\d', sent))),
        })

    feat_df = pd.DataFrame(features)

    # TF-IDF features using the global vectorizer
    tfidf = vectorizer.transform(sentences)
    feat_df['tfidf_mean'] = np.array(tfidf.mean(axis=1)).flatten()
    feat_df['tfidf_max']  = np.array(tfidf.max(axis=1).toarray()).flatten()
    feat_df['tfidf_sum']  = np.array(tfidf.sum(axis=1)).flatten()

    # Predict probabilities
    probs = model.predict_proba(feat_df[FEATURE_COLS])[:, 1]

    # Select top-N sentences, maintaining original order
    top_indices = sorted(np.argsort(probs)[-n_sentences:])
    return ' '.join([sentences[i] for i in top_indices])

# --- Test ML summarization ---
print("--- ML Summarization Example ---")
sample_article = test_df['article_clean'].iloc[0]
sample_highlights = test_df['highlights_clean'].iloc[0]

ml_summary = ml_summarize(sample_article)
print(f"\nOriginal article ({len(sample_article.split())} words):")
print(sample_article[:300] + "...\n")
print(f"Reference summary:")
print(sample_highlights)
print(f"\nML Summary ({len(ml_summary.split())} words):")
print(ml_summary)

## Model 2 — Deep Learning: Seq2Seq GRU with Bahdanau Attention

---

**Architecture:** Encoder-Decoder with Attention, built from scratch in PyTorch.

```
Article → [Embedding] → [Bidirectional GRU Encoder] → context vectors
                                                          ↓
Summary ← [FC Output] ← [GRU Decoder] ← [Attention] ← context + previous token
```

**Key components:**
1. **Encoder:** Bidirectional GRU that reads the article and produces context vectors
2. **Attention:** Bahdanau (additive) attention that lets the decoder focus on relevant parts
3. **Decoder:** GRU that generates the summary one word at a time using attention context

**Training details:**
- Vocabulary: built from training data (words with frequency ≥ 3)
- Teacher forcing: 50% probability during training
- Gradient clipping: max norm = 1.0
- Loss: CrossEntropyLoss (ignoring padding tokens)

### DL — Vocabulary & Data Preparation

In [ ]:
# ============================================================
# DL Model — Vocabulary Class
# ============================================================

class Vocabulary:
    """Word-level vocabulary with special tokens for Seq2Seq model."""

    def __init__(self, freq_threshold=VOCAB_FREQ_THRESHOLD):
        self.freq_threshold = freq_threshold
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}

    def build(self, texts):
        """Build vocabulary from a list of text strings."""
        freq = {}
        for text in texts:
            for word in text.lower().split():
                freq[word] = freq.get(word, 0) + 1

        idx = 4
        for word, count in freq.items():
            if count >= self.freq_threshold:
                self.stoi[word] = idx
                self.itos[idx] = word
                idx += 1
        print(f"Vocabulary size: {len(self.stoi):,} words")

    def encode(self, text, max_len):
        """Convert text to padded token indices."""
        tokens = [SOS_IDX]
        for word in text.lower().split()[:max_len - 2]:
            tokens.append(self.stoi.get(word, UNK_IDX))
        tokens.append(EOS_IDX)
        # Pad to max_len
        tokens += [PAD_IDX] * (max_len - len(tokens))
        return tokens

    def decode(self, indices):
        """Convert token indices back to text."""
        words = []
        for idx in indices:
            if idx == EOS_IDX:
                break
            if idx not in (PAD_IDX, SOS_IDX):
                word = self.itos.get(idx, "<UNK>")
                if word != "<UNK>":
                    words.append(word)
        return ' '.join(words)

    def __len__(self):
        return len(self.stoi)

print("Vocabulary class defined.")

In [ ]:
# ============================================================
# DL Model — Data Preparation
# ============================================================

# Use a subset for DL training (faster)
dl_train = train_df.head(DL_TRAIN_SIZE).copy()
print(f"DL training subset: {len(dl_train):,} samples")

# Build vocabulary from articles + highlights
print("\nBuilding vocabulary...")
vocab = Vocabulary(freq_threshold=VOCAB_FREQ_THRESHOLD)
all_texts = dl_train['article_clean'].tolist() + dl_train['highlights_clean'].tolist()
vocab.build(all_texts)

# Encode articles and highlights into padded sequences
print("\nEncoding sequences...")
src_data = [vocab.encode(text, MAX_ARTICLE_TOKENS)
            for text in tqdm(dl_train['article_clean'], desc="Encoding articles")]
trg_data = [vocab.encode(text, MAX_SUMMARY_TOKENS)
            for text in tqdm(dl_train['highlights_clean'], desc="Encoding highlights")]

src_tensor = torch.LongTensor(src_data)
trg_tensor = torch.LongTensor(trg_data)

print(f"\nSource tensor shape: {src_tensor.shape}")
print(f"Target tensor shape: {trg_tensor.shape}")
print(f"Example encoded article (first 20 tokens): {src_tensor[0, :20].tolist()}")
print(f"Example decoded back: {vocab.decode(src_tensor[0].tolist())[:100]}...")

# Create DataLoader
dl_dataset = TensorDataset(src_tensor, trg_tensor)
dl_loader  = DataLoader(dl_dataset, batch_size=DL_BATCH_SIZE, shuffle=True)

print(f"\nDataLoader: {len(dl_loader)} batches of size {DL_BATCH_SIZE}")

### DL — Model Architecture (Encoder, Attention, Decoder)

In [ ]:
# ============================================================
# DL Model — Neural Network Architecture
# ============================================================

class Encoder(nn.Module):
    """Bidirectional GRU Encoder."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: (batch, src_len)
        embedded = self.dropout(self.embedding(src))        # (batch, src_len, embed_dim)
        outputs, hidden = self.gru(embedded)                # outputs: (batch, src_len, hidden*2)
        # hidden: (2, batch, hidden) — combine forward & backward
        hidden = torch.tanh(self.fc(torch.cat([hidden[-2], hidden[-1]], dim=1)))
        return outputs, hidden.unsqueeze(0)                 # (1, batch, hidden)


class Attention(nn.Module):
    """Bahdanau (Additive) Attention mechanism."""

    def __init__(self, hidden_dim):
        super().__init__()
        # encoder outputs = hidden_dim*2 (bidirectional), decoder hidden = hidden_dim
        self.attn = nn.Linear(hidden_dim * 3, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: (batch, hidden_dim)
        # encoder_outputs: (batch, src_len, hidden_dim*2)
        src_len = encoder_outputs.shape[1]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)  # (batch, src_len, hidden)
        energy = torch.tanh(self.attn(torch.cat([hidden, encoder_outputs], dim=2)))
        return torch.softmax(self.v(energy).squeeze(2), dim=1)  # (batch, src_len)


class Decoder(nn.Module):
    """GRU Decoder with Attention."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.attention = Attention(hidden_dim)
        self.gru = nn.GRU(embed_dim + hidden_dim * 2, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim * 3 + embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_token, hidden, encoder_outputs):
        # input_token: (batch,)
        input_token = input_token.unsqueeze(1)                   # (batch, 1)
        embedded = self.dropout(self.embedding(input_token))     # (batch, 1, embed_dim)

        # Compute attention
        attn_weights = self.attention(hidden.squeeze(0), encoder_outputs)  # (batch, src_len)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)   # (batch, 1, hidden*2)

        # Decode
        rnn_input = torch.cat([embedded, context], dim=2)        # (batch, 1, embed+hidden*2)
        output, hidden = self.gru(rnn_input, hidden)             # output: (batch, 1, hidden)

        # Predict next word
        prediction = self.fc_out(
            torch.cat([output, context, embedded], dim=2)
        ).squeeze(1)                                             # (batch, vocab_size)
        return prediction, hidden


class Seq2Seq(nn.Module):
    """Complete Encoder-Decoder Seq2Seq model with Attention."""

    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        vocab_size = self.decoder.fc_out.out_features

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, trg_len, vocab_size).to(self.device)

        # Encode
        encoder_outputs, hidden = self.encoder(src)

        # First decoder input is the <SOS> token
        input_token = trg[:, 0]

        for t in range(1, trg_len):
            prediction, hidden = self.decoder(input_token, hidden, encoder_outputs)
            outputs[:, t] = prediction

            # Teacher forcing: use actual target or predicted token
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1)
            input_token = trg[:, t] if teacher_force else top1

        return outputs


# Instantiate the model
encoder = Encoder(len(vocab), DL_EMBED_DIM, DL_HIDDEN_DIM, DL_DROPOUT)
decoder = Decoder(len(vocab), DL_EMBED_DIM, DL_HIDDEN_DIM, DL_DROPOUT)
seq2seq_model = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in seq2seq_model.parameters() if p.requires_grad)
print(f"Seq2Seq model created on {DEVICE}")
print(f"Total trainable parameters: {total_params:,}")
print(f"\nModel architecture:")
print(seq2seq_model)

### DL — Training Loop

In [ ]:
# ============================================================
# DL Model — Training
# ============================================================

dl_optimizer = optim.Adam(seq2seq_model.parameters(), lr=DL_LR)
dl_criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

print("=" * 60)
print(f"TRAINING SEQ2SEQ MODEL — {DL_EPOCHS} EPOCHS")
print("=" * 60)

dl_train_start = time.time()
dl_losses = []

for epoch in range(DL_EPOCHS):
    seq2seq_model.train()
    epoch_loss = 0

    progress = tqdm(dl_loader, desc=f"Epoch {epoch+1}/{DL_EPOCHS}")
    for src_batch, trg_batch in progress:
        src_batch = src_batch.to(DEVICE)
        trg_batch = trg_batch.to(DEVICE)

        dl_optimizer.zero_grad()
        output = seq2seq_model(src_batch, trg_batch)

        # Reshape for loss: skip <SOS> token at position 0
        output = output[:, 1:].contiguous().view(-1, len(vocab))
        trg_flat = trg_batch[:, 1:].contiguous().view(-1)

        loss = dl_criterion(output, trg_flat)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(seq2seq_model.parameters(), 1.0)
        dl_optimizer.step()

        epoch_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = epoch_loss / len(dl_loader)
    dl_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{DL_EPOCHS} | Average Loss: {avg_loss:.4f}")

dl_train_time = time.time() - dl_train_start
print(f"\nDL Training completed in {dl_train_time:.1f} seconds")

In [ ]:
# ============================================================
# DL Model — Training Loss Curve
# ============================================================

plt.figure(figsize=(8, 4))
plt.plot(range(1, DL_EPOCHS + 1), dl_losses, 'o-', color='#ff6b6b',
         linewidth=2, markersize=8, label='Training Loss')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Seq2Seq Training Loss', fontsize=14, fontweight='bold')
plt.xticks(range(1, DL_EPOCHS + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### DL — Summarization Function & Evaluation

In [ ]:
# ============================================================
# DL Model — Summarization Function (Greedy Decoding)
# ============================================================

def dl_summarize(article, model=seq2seq_model, vocabulary=vocab,
                 device=DEVICE, max_len=MAX_SUMMARY_TOKENS):
    """
    Generate a summary using the trained Seq2Seq model.
    Uses greedy decoding (picks the highest-probability word at each step).
    """
    model.eval()

    # Encode the article
    tokens = vocabulary.encode(article, MAX_ARTICLE_TOKENS)
    src = torch.LongTensor(tokens).unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src)

        input_token = torch.LongTensor([SOS_IDX]).to(device)
        generated = []

        for _ in range(max_len):
            prediction, hidden = model.decoder(input_token, hidden, encoder_outputs)
            top1 = prediction.argmax(1)

            if top1.item() == EOS_IDX:
                break

            word = vocabulary.itos.get(top1.item(), None)
            if word and word not in ("<PAD>", "<SOS>", "<UNK>"):
                generated.append(word)
            input_token = top1

    return ' '.join(generated) if generated else "Unable to generate summary."

# --- Test DL summarization ---
print("--- DL Summarization Example ---")
dl_summary = dl_summarize(test_df['article_clean'].iloc[0])
print(f"\nReference summary:")
print(test_df['highlights_clean'].iloc[0])
print(f"\nDL Summary:")
print(dl_summary)

## Model 3 — Pre-trained: T5-small (Hugging Face)

---

### Hugging Face Model
We used here `T5 small` from google

In [ ]:
# ============================================================
# Hierarchical Long-Text Summarization with T5-small (Hugging Face)
# ============================================================
#
# How it works:
# 1. Preprocess: split the input text into sentences, then pack
#    sentences into chunks that stay under the model's 512-token
#    limit (T5's max input size)
#
# 2. Summarize each chunk with T5-small
# 3. Concatenate all chunk summaries into one combined text
# 4. Check the token count of that combined text:
#       - If it's <= 512 tokens -> run ONE final summarization pass
#         and return the result.
#       - If it's still > 512 tokens -> repeat steps 1-4 on the
#         combined summaries (i.e. summarize the summaries),
#         until the combined text fits in 512 tokens
#
# Run this in a Colab cell:
#   !pip install -q transformers sentencepiece torch
# then paste/run the rest of this file

# ---- 1. Install dependencies (run this in Colab first) ----
# !pip install -q transformers sentencepiece torch


# Constants
# NOTE: Using t5_ prefix to avoid overwriting DL model variables
T5_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "t5-small"
MAX_INPUT_TOKENS = 512
PREFIX = "summarize: "

print(f"Loading {MODEL_NAME} on {T5_DEVICE} ...")
t5_tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
t5_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(T5_DEVICE)
t5_model.eval()


def split_into_sentences(text: str):
    """Just a sentence splitter"""

    # Using Regular Expression
    text = re.sub(r"\s+", " ", text.strip())
    sentences = re.split(r"(?<=[.!?])\s+", text)

    return [s.strip() for s in sentences if s.strip()]


def count_tokens(text: str) -> int:
    return len(t5_tokenizer.encode(text, add_special_tokens=False))


def chunk_text_by_tokens(text: str, max_tokens: int = MAX_INPUT_TOKENS - 20):
    """
    Split text into chunks of sentences, each chunk staying under
    max_tokens (leaving headroom for the 'summarize: ' prefix and
    special tokens)
    """
    sentences = split_into_sentences(text)
    prefix_len = count_tokens(PREFIX)

    chunks = []
    current_chunk = []
    current_len = prefix_len

    for sentence in sentences:
        sent_len = count_tokens(sentence)

        if sent_len > max_tokens:
            if current_chunk:
                chunks.append(" ".join(current_chunk))
                current_chunk, current_len = [], prefix_len
            chunks.append(sentence)
            continue

        if current_len + sent_len > max_tokens and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_len = prefix_len + sent_len
        else:
            current_chunk.append(sentence)
            current_len += sent_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks



def summarize_chunk(text: str, max_length: int = 150, min_length: int = 30) -> str:
    """
    Core function to make a summarization.
    """
    input_text = PREFIX + text
    input_ids = t5_tokenizer.encode(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(T5_DEVICE)

    with torch.no_grad():
        summary_ids = t5_model.generate(
            input_ids,
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    return t5_tokenizer.decode(summary_ids[0], skip_special_tokens=True) + '\n'



def hierarchical_summarize(
    text: str,
    max_tokens: int = MAX_INPUT_TOKENS,
    max_length: int = 150,
    min_length: int = 30,
    max_levels: int = 6,
    verbose: bool = True,
    final_summary_pass: bool = False,
) -> str:
    """
    Repeatedly chunk + summarize until the combined summary text
    fits within max_tokens, then run one final summarization pass.
    """
    current_text = text
    level = 0

    while True:
        token_count = count_tokens(current_text)
        if verbose:
            print(f"[Level {level}] current text = {token_count} tokens")

        if token_count <= max_tokens:
            if not final_summary_pass:
                if verbose:
                    print(f"[Level {level}] fits in {max_tokens} tokens -> "
                          f"final_summary_pass=False, returning as-is.")
                return current_text

            # Core Model
            final_summary = summarize_chunk(
                current_text, max_length=max_length, min_length=min_length
            )
            if verbose:
                print(f"[Level {level}] fits in {max_tokens} tokens -> final summary produced.")
            return final_summary

        if level >= max_levels:
            if verbose:
                print("Max recursion depth reached, forcing a final pass with truncation.")
            return summarize_chunk(current_text, max_length=max_length, min_length=min_length)

        # Recursive case: chunk, summarize each chunk, merge, repeat
        chunks = chunk_text_by_tokens(current_text, max_tokens=max_tokens - 20)
        if verbose:
            print(f"[Level {level}] split into {len(chunks)} chunk(s)")

        chunk_summaries = []
        for i, chunk in enumerate(chunks):
            if verbose:
                print(f"  -> summarizing chunk {i + 1}/{len(chunks)} "
                      f"({count_tokens(chunk)} tokens)")
            chunk_summaries.append(
                summarize_chunk(chunk, max_length=max_length, min_length=min_length)
            )

        current_text = " ".join(chunk_summaries)
        level += 1

### Testing model with single real example

In [ ]:
large_text = """
The evolution of artificial intelligence from a theoretical concept to a foundational pillar of modern technology represents one of the most remarkable trajectories in human history, fundamentally reshaping how we interact with machines, process vast swaths of digital information, and conceptualize the very nature of cognition. In the early days of computing, pioneers like Alan Turing and John von Neumann laid the philosophical and mathematical groundwork, posing the enduring question of whether machines could truly think, which eventually birthed the historic Dartmouth workshop in the 1950s where the term "artificial intelligence" was officially coined. For several decades, the field experienced a volatile series of booms and busts, commonly referred to as AI winters, where initial optimism and ambitious promises were continually met with the harsh realities of limited computational power, symbolic logic constraints, and vastly insufficient datasets. However, the dawn of the 21st century brought about a monumental paradigm shift driven largely by exponential advancements in computer engineering, specifically the ingenious repurposing of graphical processing units, or GPUs. Originally designed strictly for rendering complex polygons in video games, these GPUs proved exceptionally adept at handling the massive, highly parallel matrix multiplications required by artificial neural networks. This hardware revolution, coupled seamlessly with the explosion of unstructured digital data generated by the global internet, provided the necessary computational fuel for deep learning algorithms to achieve unprecedented breakthroughs in complex domains that were previously thought to be the exclusive domain of human intelligence. Consequently, modern programming languages, most notably Python, rapidly emerged as the undisputed lingua franca of this new technological era, primarily due to its highly readable, intuitive syntax and the subsequent development of incredibly robust, open-source computational libraries. Frameworks such as TensorFlow, PyTorch, Keras, and Scikit-Learn essentially abstracted away the immensely complex underlying calculus and linear algebra, allowing a new generation of software engineers and researchers to focus their cognitive efforts entirely on model architecture, hyperparameter optimization, and creative deployment rather than writing foundational math from scratch. As these powerful tools became democratized and widely accessible to the global open-source community, the diverse subfields of machine learning expanded at a breakneck pace, with natural language processing (NLP) experiencing a literal renaissance thanks to the groundbreaking invention of the Transformer network architecture. By dispensing with traditional recurrent sequential processing in favor of innovative self-attention mechanisms, Transformers enabled systems to seamlessly understand context, nuance, and semantic relationships across entire documents simultaneously, directly leading to the creation of massive Large Language Models (LLMs) capable of drafting complex academic essays, writing and debugging functional software code, and engaging in deeply nuanced, context-aware dialogue with human users. Concurrently, the field of computer vision was entirely revolutionized by deep convolutional neural networks (CNNs) that empowered autonomous vehicles to safely navigate chaotic urban environments by recognizing pedestrians and traffic signals in real-time, enabled cutting-edge medical software to detect microscopic anomalies in radiological scans with superhuman accuracy, and allowed facial recognition systems to be deployed on a global scale. Yet, it is vital to recognize that all of these software marvels are inextricably linked to and fundamentally limited by physical computer engineering, as the insatiable, exponential demand for computational resources has spurred the rapid design of highly specialized, application-specific silicon. Innovations such as Google's Tensor Processing Units (TPUs) and experimental neuromorphic chips designed explicitly to mimic the biological structure and energy efficiency of the human brain aim to drastically reduce the massive, unsustainable energy footprint currently required to train state-of-the-art AI models in warehouse-sized data centers. Furthermore, the aggressive industry push towards edge computing is driving hardware engineers to ruthlessly optimize and compress these gargantuan neural networks so they can execute locally on low-power smartphones, autonomous drones, and embedded internet-of-things (IoT) devices, thereby preserving critical user privacy, eliminating network latency, and bypassing the constant need for cloud connectivity. This intricate, symbiotic dance between advancing hardware capabilities and software ingenuity is continuously pushing the boundaries of what is technologically possible, yet it simultaneously introduces profound ethical, legal, and societal challenges that engineers, ethicists, and policymakers must navigate with extreme caution. These pressing challenges include the dangerous amplification of historical human biases inevitably present in massive web-scraped training datasets, the looming socioeconomic threat of automating millions of white-collar and analytical jobs, the weaponization of generative AI to create hyper-realistic deepfakes that threaten the very fabric of objective truth, and the incredibly complex alignment problem of ensuring that highly autonomous, super-intelligent systems act strictly in accordance with human values and safety parameters. To systematically mitigate these existential risks, the global research community is increasingly focusing vast resources on the subfield of explainable AI (XAI), which seeks to crack open the opaque "black box" of deep learning models so that their internal decision-making processes can be rigorously audited, mapped, and mathematically understood by human operators—an absolute critical requirement for the legal deployment of AI in high-stakes domains like emergency healthcare, criminal justice sentencing, and autonomous mass transit. Looking even further ahead, the impending intersection of artificial intelligence with emerging, radical paradigms such as quantum computing promises yet another explosive, unpredictable leap in computational capabilities, as theoretical quantum algorithms could eventually optimize complex neural networks exponentially faster than today's most advanced classical supercomputers, potentially unlocking the ultimate holy grail of computer science: Artificial General Intelligence (AGI), a hypothetical, self-improving system that equals or far exceeds human intellectual capacity across all conceivable cognitive and economic tasks. Achieving such a monumental milestone will strictly require an unprecedented, unified synthesis of scientific disciplines, relying not just on software developers writing elegant Python scripts, but deeply depending on computer engineers to design the novel atomic-scale architectures, revolutionary memory hierarchies, and massive photonic interconnects capable of sustaining such immense computational loads without literally melting the silicon that houses them. As humanity stands precariously on the precipice of this new technological epoch, it is abundantly clear that artificial intelligence is not merely a transient software trend, but rather a permanent, foundational infrastructure layer that will irrevocably dictate the future trajectory of human civilization, demanding a rigorous, multidisciplinary approach to ensure it is developed safely, equitably, and in a manner that actively augments rather than diminishes the human experience, ultimately requiring a new generation of visionary engineers who are as flawlessly fluent in the ethical implications of their creations as they are in the granular intricacies of backpropagation algorithms and silicon wafer fabrication.
"""

result = hierarchical_summarize(
    large_text,
    max_tokens=512,
    max_length=150,
    min_length=30,
    verbose=True,
)

print("\n===== FINAL SUMMARY =====")
print(count_tokens(large_text))
print(count_tokens(result))
print(result)

## Performance Comparison

---

Evaluating all three models on the **same test subset** using ROUGE metrics:
- **ROUGE-1:** Overlap of unigrams (single words)
- **ROUGE-2:** Overlap of bigrams (word pairs)
- **ROUGE-L:** Longest common subsequence

In [ ]:
# ============================================================
# Evaluation — Shared Helper Function
# ============================================================

def evaluate_model(summarize_fn, test_data, n_samples=N_EVAL_SAMPLES, desc="Evaluating"):
    """Evaluate a summarization function using ROUGE scores on the test set."""
    scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    r1, r2, rl = [], [], []

    for _, row in tqdm(test_data.head(n_samples).iterrows(),
                       total=min(n_samples, len(test_data)), desc=desc):
        try:
            summary = summarize_fn(row['article_clean'])
            if not summary or len(summary.strip()) == 0:
                summary = "empty summary"
            scores = scorer_obj.score(row['highlights_clean'], summary)
            r1.append(scores['rouge1'].fmeasure)
            r2.append(scores['rouge2'].fmeasure)
            rl.append(scores['rougeL'].fmeasure)
        except Exception as e:
            continue

    return {
        'ROUGE-1': np.mean(r1) if r1 else 0,
        'ROUGE-2': np.mean(r2) if r2 else 0,
        'ROUGE-L': np.mean(rl) if rl else 0,
    }

print("Evaluation helper defined.")

In [ ]:
# ============================================================
# Evaluate All Three Models
# ============================================================

print("=" * 60)
print(f"EVALUATING ALL MODELS ON {N_EVAL_SAMPLES} TEST SAMPLES")
print("=" * 60)

# --- 1. ML Model Evaluation ---
print("\n[1/3] Evaluating ML Model (RandomForest Extractive)...")
ml_eval_start = time.time()
ml_rouge = evaluate_model(
    lambda text: ml_summarize(text, rf_model, tfidf_vectorizer),
    test_df, N_EVAL_SAMPLES, desc="ML Eval"
)
ml_eval_time = time.time() - ml_eval_start
print(f"  ROUGE: {ml_rouge}")
print(f"  Eval time: {ml_eval_time:.1f}s")

# --- 2. DL Model Evaluation ---
print("\n[2/3] Evaluating DL Model (Seq2Seq with Attention)...")
dl_eval_start = time.time()
dl_rouge = evaluate_model(
    lambda text: dl_summarize(text, seq2seq_model, vocab, DEVICE),
    test_df, N_EVAL_SAMPLES, desc="DL Eval"
)
dl_eval_time = time.time() - dl_eval_start
print(f"  ROUGE: {dl_rouge}")
print(f"  Eval time: {dl_eval_time:.1f}s")

# --- 3. T5 Model Evaluation ---
print("\n[3/3] Evaluating T5 Model (Hugging Face Pre-trained)...")
print("  (This may take several minutes due to beam search decoding)")
t5_eval_start = time.time()
t5_rouge = evaluate_model(
    lambda text: hierarchical_summarize(text, verbose=False),
    test_df, N_EVAL_SAMPLES, desc="T5 Eval"
)
t5_eval_time = time.time() - t5_eval_start
print(f"  ROUGE: {t5_rouge}")
print(f"  Eval time: {t5_eval_time:.1f}s")

print("\n" + "=" * 60)
print("EVALUATION COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================
# Performance Comparison — Table & Visualization
# ============================================================

# Build comparison table
results_df = pd.DataFrame({
    'Model': ['ML (RandomForest)', 'DL (Seq2Seq GRU)', 'T5 (Pre-trained)'],
    'ROUGE-1': [ml_rouge['ROUGE-1'], dl_rouge['ROUGE-1'], t5_rouge['ROUGE-1']],
    'ROUGE-2': [ml_rouge['ROUGE-2'], dl_rouge['ROUGE-2'], t5_rouge['ROUGE-2']],
    'ROUGE-L': [ml_rouge['ROUGE-L'], dl_rouge['ROUGE-L'], t5_rouge['ROUGE-L']],
    'Train Time (s)': [ml_train_time, dl_train_time, 0],  # T5 is pre-trained, no training
    'Eval Time (s)': [ml_eval_time, dl_eval_time, t5_eval_time],
})

print("\n" + "=" * 60)
print("PERFORMANCE COMPARISON TABLE")
print("=" * 60)
print(results_df.to_string(index=False, float_format='%.4f'))

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. ROUGE Scores — Grouped Bar Chart
rouge_data = results_df[['Model', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']].melt(
    id_vars='Model', var_name='Metric', value_name='Score'
)
palette = {'ML (RandomForest)': '#00d4aa', 'DL (Seq2Seq GRU)': '#ff6b6b', 'T5 (Pre-trained)': '#4ecdc4'}
sns.barplot(data=rouge_data, x='Metric', y='Score', hue='Model',
            ax=axes[0], palette=palette)
axes[0].set_title('ROUGE Scores Comparison', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].legend(fontsize=8, loc='upper right')

# Add value labels on bars
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.3f', fontsize=7, padding=2)

# 2. Training Time
colors = ['#00d4aa', '#ff6b6b', '#4ecdc4']
bars = axes[1].bar(results_df['Model'], results_df['Train Time (s)'], color=colors, edgecolor='white')
axes[1].set_title('Training Time', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Time (seconds)')
axes[1].tick_params(axis='x', rotation=15)
for bar in bars:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}s', ha='center', va='bottom', fontsize=10)

# 3. Evaluation Time
bars2 = axes[2].bar(results_df['Model'], results_df['Eval Time (s)'], color=colors, edgecolor='white')
axes[2].set_title('Evaluation Time', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Time (seconds)')
axes[2].tick_params(axis='x', rotation=15)
for bar in bars2:
    height = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                f'{height:.1f}s', ha='center', va='bottom', fontsize=10)

plt.suptitle('Model Performance Comparison — CNN/DailyMail Summarization',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Analysis

---

**Key Observations:**
- **ML (RandomForest):** Fast training and inference. Uses extractive approach, so summaries are always grammatically correct (they are original sentences). Limited by sentence selection granularity.
- **DL (Seq2Seq):** Moderate training time. Abstractive approach generates new text, but limited vocabulary and short training leads to lower quality compared to the pre-trained model.
- **T5 (Pre-trained):** No training required (uses pre-trained weights). Highest ROUGE scores due to massive pre-training on diverse text data. Slowest inference due to beam search decoding.

**Trade-offs:** ML is fastest but least flexible. DL offers custom architectures but needs large data/compute. Pre-trained models give best quality with zero training but require more inference resources.

### Side-by-Side Comparison Examples

In [ ]:
# ============================================================
# Side-by-Side Summary Comparison on Test Articles
# ============================================================

print("=" * 70)
print("SIDE-BY-SIDE COMPARISON — 3 TEST ARTICLES")
print("=" * 70)

for i in range(3):
    article = test_df['article_clean'].iloc[i]
    reference = test_df['highlights_clean'].iloc[i]

    print(f"\n{'='*70}")
    print(f"ARTICLE {i+1} (first 200 chars):")
    print(f"{'='*70}")
    print(article[:200] + "...")

    print(f"\n📌 Reference Summary:")
    print(f"   {reference[:200]}")

    print(f"\n🤖 ML Summary (RandomForest):")
    ml_s = ml_summarize(article, rf_model, tfidf_vectorizer)
    print(f"   {ml_s[:200]}")

    print(f"\n🧠 DL Summary (Seq2Seq):")
    dl_s = dl_summarize(article, seq2seq_model, vocab, DEVICE)
    print(f"   {dl_s[:200]}")

    print(f"\n🔥 T5 Summary (Pre-trained):")
    t5_s = hierarchical_summarize(article, verbose=False)
    print(f"   {t5_s[:200]}")

## Deployment — API & User Interface

---

### Gradio UI

An interactive web interface with tabs for each model. Run the cell below and click the link to open the UI in your browser.

In [ ]:
# ============================================================
# Gradio UI — Interactive Summarization Interface
# ============================================================

import gradio as gr

def gradio_ml_summarize(text, n_sentences):
    """Gradio wrapper for ML summarization."""
    if not text or len(text.strip()) < 20:
        return "Please enter a longer article to summarize."
    return ml_summarize(text, rf_model, tfidf_vectorizer, n_sentences=int(n_sentences))

def gradio_dl_summarize(text):
    """Gradio wrapper for DL summarization."""
    if not text or len(text.strip()) < 20:
        return "Please enter a longer article to summarize."
    return dl_summarize(text, seq2seq_model, vocab, DEVICE)

def gradio_t5_summarize(text):
    """Gradio wrapper for T5 summarization."""
    if not text or len(text.strip()) < 20:
        return "Please enter a longer article to summarize."
    return hierarchical_summarize(text, max_tokens=512, max_length=150, min_length=30, verbose=False)

# Sample article for quick testing
sample_text = test_df['article_clean'].iloc[5]

# Build the Gradio interface
with gr.Blocks(
    title="Text Summarization — CNN/DailyMail",
    theme=gr.themes.Soft(primary_hue="teal", secondary_hue="rose")
) as demo:

    gr.Markdown("# 📝 Text Summarization Pipeline")
    gr.Markdown("Summarize news articles using three different approaches: ML, Deep Learning, and T5 Pre-trained model.")

    with gr.Tab("🤖 ML Model (RandomForest)"):
        gr.Markdown("**Extractive summarization** — selects the most important sentences from the article.")
        with gr.Row():
            with gr.Column():
                ml_input = gr.Textbox(label="Input Article", lines=10,
                                      placeholder="Paste a news article here...",
                                      value=sample_text[:1000])
                ml_n = gr.Slider(1, 10, value=3, step=1, label="Number of sentences to extract")
                ml_btn = gr.Button("Summarize", variant="primary")
            with gr.Column():
                ml_output = gr.Textbox(label="ML Summary", lines=8)
        ml_btn.click(gradio_ml_summarize, [ml_input, ml_n], ml_output)

    with gr.Tab("🧠 DL Model (Seq2Seq)"):
        gr.Markdown("**Abstractive summarization** — generates new text using Encoder-Decoder with Attention.")
        with gr.Row():
            with gr.Column():
                dl_input = gr.Textbox(label="Input Article", lines=10,
                                      placeholder="Paste a news article here...",
                                      value=sample_text[:1000])
                dl_btn = gr.Button("Summarize", variant="primary")
            with gr.Column():
                dl_output = gr.Textbox(label="DL Summary", lines=8)
        dl_btn.click(gradio_dl_summarize, dl_input, dl_output)

    with gr.Tab("🔥 T5 Model (Pre-trained)"):
        gr.Markdown("**Abstractive summarization** — uses Google's T5-small with hierarchical chunking.")
        with gr.Row():
            with gr.Column():
                t5_input = gr.Textbox(label="Input Article", lines=10,
                                      placeholder="Paste a news article here...",
                                      value=sample_text[:1000])
                t5_btn = gr.Button("Summarize", variant="primary")
            with gr.Column():
                t5_output = gr.Textbox(label="T5 Summary", lines=8)
        t5_btn.click(gradio_t5_summarize, t5_input, t5_output)

print("Launching Gradio interface...")
demo.launch(share=False, inline=True)

### FastAPI — REST API

---

Below is a FastAPI server that exposes summarization endpoints for all three models.
The code is defined inline — run the cell to start the server, then use curl or any HTTP client to query it.

**Endpoints:**
- `POST /summarize` — Takes JSON with `text`, `model` ("ml", "dl", or "t5"), and optional `n_sentences`

**Example curl request:**
```bash
curl -X POST http://localhost:8000/summarize \
  -H "Content-Type: application/json" \
  -d '{"text": "Your article text here...", "model": "t5"}'
```

**Example curl for ML model with custom sentence count:**
```bash
curl -X POST http://localhost:8000/summarize \
  -H "Content-Type: application/json" \
  -d '{"text": "Your article text here...", "model": "ml", "n_sentences": 5}'
```

In [ ]:
# ============================================================
# FastAPI — Summarization API (inline)
# ============================================================
# Uncomment and run this cell to start the API server.
# Note: This will block the notebook — use Gradio above for interactive use.

# from fastapi import FastAPI
# from pydantic import BaseModel
# import uvicorn
# import threading
#
# app = FastAPI(title="Text Summarization API", version="1.0")
#
# class SumRequest(BaseModel):
#     text: str
#     model: str = "t5"          # "ml", "dl", or "t5"
#     n_sentences: int = 3       # Only for ML model
#
# class SumResponse(BaseModel):
#     summary: str
#     model_used: str
#
# @app.post("/summarize", response_model=SumResponse)
# async def summarize_endpoint(req: SumRequest):
#     if req.model == "ml":
#         summary = ml_summarize(req.text, rf_model, tfidf_vectorizer, n_sentences=req.n_sentences)
#     elif req.model == "dl":
#         summary = dl_summarize(req.text, seq2seq_model, vocab, DEVICE)
#     else:
#         summary = hierarchical_summarize(req.text, verbose=False)
#     return SumResponse(summary=summary, model_used=req.model)
#
# @app.get("/health")
# async def health():
#     return {"status": "ok", "models": ["ml", "dl", "t5"]}
#
# # Run in a background thread so the notebook doesn't block
# def run_api():
#     uvicorn.run(app, host="0.0.0.0", port=8000)
#
# api_thread = threading.Thread(target=run_api, daemon=True)
# api_thread.start()
# print("FastAPI server started at http://localhost:8000")
# print("API docs available at http://localhost:8000/docs")

print("FastAPI code is ready — uncomment the cell above to start the server.")
print("Use the Gradio interface above for interactive testing.")

## Summary

---

This notebook implemented a complete **text summarization pipeline** on the CNN/DailyMail dataset with three approaches:

| Aspect | ML (RandomForest) | DL (Seq2Seq GRU) | T5 (Pre-trained) |
|--------|-------------------|-------------------|-------------------|
| **Type** | Extractive | Abstractive | Abstractive |
| **Training** | Feature engineering + classifier | End-to-end neural training | No training (pre-trained) |
| **Strengths** | Fast, grammatical output | Custom architecture, flexible | Best quality, no training needed |
| **Weaknesses** | Limited to existing sentences | Needs lots of data/compute | Slow inference, large model |
| **Best For** | Quick extraction, limited resources | Research, custom domains | Production quality summaries |

All models are accessible via the **Gradio UI** for interactive testing and via **FastAPI** for programmatic access.